[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Connections and Transactions &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: the engine, the statements `ADD` and `DROP`, `spring_courses`, `SPRING_2026`,
`apply_moves` and the `EVENING` batch. Run it first. The tasks do not depend on one another, and the
last cell removes the scratch folder.


In [1]:
import logging
import shutil
import sqlite3
from pathlib import Path

import sqlalchemy
from sqlalchemy import create_engine, event, text
from sqlalchemy.exc import IntegrityError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL);
    CREATE TABLE sections (id INTEGER PRIMARY KEY, course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id), capacity INTEGER NOT NULL);
    CREATE TABLE enrollments (student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL, grade TEXT,
                              PRIMARY KEY (student_id, section_id));
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.executemany("INSERT INTO terms (name, starts_on) VALUES (?, ?)", TERMS)
build.executemany("INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)", SECTIONS)
build.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)", ENROLLMENTS)
build.commit()
build.close()

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


ADD = text("INSERT INTO enrollments (student_id, section_id, status) VALUES (:student, :new, 'enrolled')")
DROP = text("DELETE FROM enrollments WHERE student_id = :student AND section_id = :old")
SPRING_COURSES = text("""
    SELECT courses.code FROM enrollments
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    WHERE enrollments.student_id = :student AND sections.term_id = 4
    ORDER BY courses.code
""")
SPRING_2026 = {code: 30 + number for number, (code, *_) in enumerate(COURSES, start=1)}   # course code -> section


def spring_courses(conn, student):
    """The codes of the courses a student takes in Spring 2026."""
    return conn.execute(SPRING_COURSES, {"student": student}).scalars().all()


engine = college_engine(DATABASE)


class NotEnrolled(Exception):
    """The student in a move is not enrolled in the section they are moving out of."""


def apply_moves(engine, moves):
    """Apply a batch of section moves, each one all or nothing, and return the moves refused, with the reason."""
    refused = []
    with engine.begin() as conn:
        for change in moves:
            try:
                with conn.begin_nested():
                    if conn.execute(DROP, change).rowcount == 0:
                        raise NotEnrolled("not enrolled in the old section")
                    conn.execute(ADD, change)
            except IntegrityError as error:
                refused.append((change, str(error.orig)))
            except NotEnrolled as error:
                refused.append((change, str(error)))
    return refused


EVENING = [
    {"student": 9, "old": SPRING_2026["PSY-101"], "new": SPRING_2026["STA-200"]},    # Isabel Costa
    {"student": 10, "old": SPRING_2026["ENG-105"], "new": SPRING_2026["STA-200"]},   # Jonas Berg, already in Statistics
    {"student": 11, "old": SPRING_2026["MAT-120"], "new": SPRING_2026["MAT-121"]},   # Keiko Tanaka, not in Calculus I
    {"student": 12, "old": SPRING_2026["CHE-110"], "new": 41},                       # Liam Murphy, to no such section
    {"student": 13, "old": SPRING_2026["ENG-105"], "new": SPRING_2026["HIS-110"]},   # Maya Patel
]

print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "holds", len(ENROLLMENTS), "enrollments")


sqlalchemy 2.0.54 | scratch/college.db holds 228 enrollments


**1.** Commit as you go.


In [2]:
with engine.connect() as conn:
    conn.execute(ADD, {"student": 14, "new": SPRING_2026["STA-200"]})       # Noah Andersen adds Statistics
    conn.commit()

with engine.connect() as conn:
    print("Noah Andersen:", spring_courses(conn, 14))


Noah Andersen: ['BIO-101', 'HIS-110', 'MAT-121', 'STA-200']


The commit saved the change before the block ended, so the rollback at the end had nothing left to
undo, and a new connection sees Statistics.


**2.** When a connection is in a transaction.


In [3]:
with engine.connect() as conn:
    print("before the first statement:", conn.in_transaction())
    conn.execute(text("SELECT COUNT(*) FROM enrollments"))
    print("after it:                  ", conn.in_transaction())
    conn.commit()
    print("after commit():            ", conn.in_transaction())


before the first statement: False
after it:                   True
after commit():             False


The first statement began the transaction, which is autobegin, and `commit()` ended it. The next
statement would begin another.


**3.** An exception of your own inside `begin()`.


In [4]:
try:
    with engine.begin() as conn:
        conn.execute(ADD, {"student": 15, "new": SPRING_2026["STA-200"]})   # Olivia Brandt adds Statistics
        raise ValueError("the advisor has not signed off")
except ValueError as error:
    print("stopped:", error)

with engine.connect() as conn:
    print("Olivia Brandt:", spring_courses(conn, 15))


stopped: the advisor has not signed off
Olivia Brandt: ['CHE-110', 'CSC-101', 'PSY-101']


The database accepted the `INSERT`, and the block still rolled it back, because an exception left
the block. `begin()` does not care who raised: any exception means the work is unfinished.


**4.** `rowcount`, in a block that never commits.


In [5]:
with engine.connect() as conn:
    dropped = conn.execute(text("DELETE FROM enrollments WHERE student_id = 16 AND section_id > 30"))
    print("rows removed:", dropped.rowcount)
    print("during the block:", spring_courses(conn, 16))

with engine.connect() as conn:
    print("after the block: ", spring_courses(conn, 16))


rows removed: 3
during the block: []
after the block:  ['CSC-201', 'MAT-120', 'STA-200']


Pavel Novak's three Spring 2026 sections have the ids above 30, so one `DELETE` removed three rows,
and the block's rollback put all three back.


**5.** Two savepoints, one refused.


In [6]:
with engine.begin() as conn:
    for section in (SPRING_2026["STA-200"], 41):
        try:
            with conn.begin_nested():
                conn.execute(ADD, {"student": 17, "new": section})          # Quinn Harper
            print("section", section, "added")
        except IntegrityError as error:
            print("section", section, "refused:", error.orig)

with engine.connect() as conn:
    print("Quinn Harper:", spring_courses(conn, 17))


section 40 added
section 41 refused: FOREIGN KEY constraint failed
Quinn Harper: ['BIO-101', 'ENG-105', 'MAT-121', 'STA-200']


The refused savepoint took back only its own `INSERT`, and the `COMMIT` at the end of the block saved
Statistics.


**6.** The same batch, a second time.


In [7]:
first_run = apply_moves(engine, EVENING)
print("the first run refused", len(first_run), "of", len(EVENING))
for change, reason in apply_moves(engine, EVENING):
    print("the second refused:", NAMES[change["student"] - 1], "|", reason)


the first run refused 3 of 5
the second refused: Isabel Costa | not enrolled in the old section
the second refused: Jonas Berg | UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
the second refused: Keiko Tanaka | not enrolled in the old section
the second refused: Liam Murphy | FOREIGN KEY constraint failed
the second refused: Maya Patel | not enrolled in the old section


The second run refuses all five. Isabel Costa and Maya Patel moved the first time, so they are no
longer in the sections they would move out of, and `rowcount` finds nothing to delete. The other
three are refused for the same reasons as before. A batch that is run twice by mistake changes
nothing the second time, which is worth having in a job that runs every evening.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Connections and Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/03-connections-and-transactions.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
